# IUCN RLE Assessments Scraper
Scrapes assessments from `https://assessments.iucnrle.org/assessments/1` through `/550`

In [26]:
# Run once to install Playwright's Chromium browser
import subprocess
result = subprocess.run(["playwright", "install", "chromium"], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

In [3]:
import asyncio
import json
import time
import pandas as pd
from pathlib import Path
from playwright.async_api import async_playwright

In [8]:
BASE_URL = "https://assessments.iucnrle.org/assessments"
START_ID = 1
END_ID = 600
OUTPUT_DIR = Path("data/iucn-rle-database")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE = OUTPUT_DIR / "iucnrle_assessments.json"

# Delay in seconds after page load before scraping
PAGE_LOAD_WAIT = 3.0
# Delay between requests (be polite)
REQUEST_DELAY = 1.5

In [9]:
async def extract_text_list(page, label_text):
    """
    Find a heading/label on the page and extract all bullet/sub-items under it.
    Returns a list of strings.
    """
    try:
        # Try to find a heading that contains the label text
        items = await page.evaluate("""
            (labelText) => {
                // Find all elements that contain the label text
                const allEls = Array.from(document.querySelectorAll('*'));
                const header = allEls.find(el => 
                    el.children.length === 0 && 
                    el.textContent.trim() === labelText
                );
                if (!header) return null;
                
                // Walk up to find a container, then get sibling/child content
                let container = header.parentElement;
                // Collect all text nodes in siblings/children after the header
                const results = [];
                
                // Get the parent's children and find text after the header
                const siblings = Array.from(container.children);
                let found = false;
                for (const sib of siblings) {
                    if (sib === header) { found = true; continue; }
                    if (found) {
                        const txt = sib.textContent.trim();
                        if (txt) results.push(txt);
                    }
                }
                return results.length > 0 ? results : [container.textContent.replace(labelText, '').trim()];
            }
        """, label_text)
        return items if items else []
    except Exception:
        return []


async def extract_field_value(page, label_text):
    """Extract a single value that follows a label."""
    try:
        value = await page.evaluate("""
            (labelText) => {
                const allEls = Array.from(document.querySelectorAll('*'));
                const label = allEls.find(el =>
                    el.children.length === 0 &&
                    el.textContent.trim() === labelText
                );
                if (!label) return null;
                // Next sibling or parent's next sibling
                const parent = label.parentElement;
                const siblings = Array.from(parent.parentElement.children);
                const idx = siblings.indexOf(parent);
                if (idx >= 0 && idx + 1 < siblings.length) {
                    return siblings[idx + 1].textContent.trim();
                }
                return null;
            }
        """, label_text)
        return value
    except Exception:
        return None


async def scrape_assessment(page, assessment_id):
    url = f"{BASE_URL}/{assessment_id}"
    result = {"id": assessment_id, "link": url}
    
    try:
        response = await page.goto(url, wait_until="networkidle", timeout=30000)
        
        # Check for 404 / not found
        if response and response.status == 404:
            result["status"] = "not_found"
            return result
        
        # Wait additional time for JS to render dynamic content
        await asyncio.sleep(PAGE_LOAD_WAIT)
        
        # Check if page has content (look for a known element)
        title_el = await page.query_selector('h1, h2, [class*="title"]')
        if not title_el:
            result["status"] = "no_content"
            return result
        
        result["status"] = "ok"
        
        # ── Citation ──
        citation = await page.evaluate("""
            () => {
                const allEls = Array.from(document.querySelectorAll('*'));
                const label = allEls.find(el =>
                    el.children.length === 0 &&
                    el.textContent.trim() === 'Citation:'
                );
                if (!label) return null;
                const parent = label.parentElement;
                // The citation text is usually the next sibling element
                const next = parent.nextElementSibling;
                return next ? next.textContent.trim() : parent.textContent.replace('Citation:', '').trim();
            }
        """)
        result["citation"] = citation
        
        # ── IUCN Habitats Classification Scheme ──
        habitats = await page.evaluate("""
            () => {
                const allEls = Array.from(document.querySelectorAll('*'));
                const label = allEls.find(el =>
                    el.children.length === 0 &&
                    el.textContent.trim() === 'IUCN Habitats Classification Scheme'
                );
                if (!label) return [];
                const container = label.closest('div, section, td, th') || label.parentElement;
                // Collect all text lines in the container (skip the header itself)
                const lines = [];
                const walker = document.createTreeWalker(container, NodeFilter.SHOW_TEXT);
                let node;
                while ((node = walker.nextNode())) {
                    const txt = node.textContent.trim();
                    if (txt && txt !== 'IUCN Habitats Classification Scheme') {
                        lines.push(txt);
                    }
                }
                return lines;
            }
        """)
        result["iucn_habitats"] = habitats
        
        # ── IUCN Global Ecosystem Typology ──
        typology = await page.evaluate("""
            () => {
                const allEls = Array.from(document.querySelectorAll('*'));
                const label = allEls.find(el =>
                    el.children.length === 0 &&
                    el.textContent.trim() === 'IUCN Global Ecosystem Typology'
                );
                if (!label) return [];
                const container = label.closest('div, section, td, th') || label.parentElement;
                const lines = [];
                const walker = document.createTreeWalker(container, NodeFilter.SHOW_TEXT);
                let node;
                while ((node = walker.nextNode())) {
                    const txt = node.textContent.trim();
                    if (txt && txt !== 'IUCN Global Ecosystem Typology') {
                        lines.push(txt);
                    }
                }
                return lines;
            }
        """)
        result["iucn_typology"] = typology
        
        # ── Distribution metadata fields ──
        # System, Biogeographic Realm, Geographic Region, Countries
        dist_fields = await page.evaluate("""
            () => {
                const FIELD_LABELS = ['System', 'Biogeographic Realm', 'Geographic Region', 'Countries'];
                const result = {};
                
                for (const fieldLabel of FIELD_LABELS) {
                    // Find the label element
                    const allEls = Array.from(document.querySelectorAll('*'));
                    const label = allEls.find(el =>
                        el.children.length === 0 &&
                        el.textContent.trim() === fieldLabel
                    );
                    if (!label) { result[fieldLabel] = null; continue; }
                    
                    const parent = label.parentElement;
                    // Collect all leaf text in this parent (excluding the label itself)
                    const values = [];
                    const walker = document.createTreeWalker(parent, NodeFilter.SHOW_TEXT);
                    let node;
                    while ((node = walker.nextNode())) {
                        const txt = node.textContent.trim();
                        if (txt && txt !== fieldLabel) values.push(txt);
                    }
                    // Next sibling may hold the value
                    if (values.length === 0) {
                        const next = parent.nextElementSibling;
                        if (next) {
                            const nextWalker = document.createTreeWalker(next, NodeFilter.SHOW_TEXT);
                            let n;
                            while ((n = nextWalker.nextNode())) {
                                const t = n.textContent.trim();
                                if (t) values.push(t);
                            }
                        }
                    }
                    result[fieldLabel] = values.length === 1 ? values[0] : values;
                }
                return result;
            }
        """)
        result["system"] = dist_fields.get("System")
        result["biogeographic_realm"] = dist_fields.get("Biogeographic Realm")
        result["geographic_region"] = dist_fields.get("Geographic Region")
        result["countries"] = dist_fields.get("Countries")
        if isinstance(result["countries"], str):
            result["countries"] = [result["countries"]]
        
        # ── Taxa ──
        taxa = await page.evaluate("""
            () => {
                const allEls = Array.from(document.querySelectorAll('*'));
                const label = allEls.find(el =>
                    el.children.length === 0 &&
                    el.textContent.trim() === 'Taxa'
                );
                if (!label) return [];
                // Taxa items are usually italic/em children next to the label
                const container = label.closest('div, section, li') || label.parentElement;
                // Look for italic (em) or specific styled children
                const items = [];
                // Collect leaf text nodes, excluding 'Taxa' label
                const walker = document.createTreeWalker(container, NodeFilter.SHOW_TEXT);
                let node;
                while ((node = walker.nextNode())) {
                    const txt = node.textContent.trim();
                    if (txt && txt !== 'Taxa') items.push(txt);
                }
                return items;
            }
        """)
        result["taxa"] = taxa
        
    except Exception as e:
        result["status"] = f"error: {str(e)}"
    
    return result

In [10]:
async def run_scraper(start=START_ID, end=END_ID, resume=True):
    """
    Main scraper. Set resume=True to skip already-scraped IDs.
    Results are saved incrementally to OUTPUT_FILE.
    """
    # Load existing results if resuming
    existing = {}
    if resume and OUTPUT_FILE.exists():
        with open(OUTPUT_FILE) as f:
            for item in json.load(f):
                existing[item["id"]] = item
        print(f"Loaded {len(existing)} existing records, resuming...")
    
    all_results = dict(existing)
    ids_to_scrape = [i for i in range(start, end + 1) if i not in all_results]
    print(f"IDs to scrape: {len(ids_to_scrape)}")
    
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36"
        )
        page = await context.new_page()
        
        for i, assessment_id in enumerate(ids_to_scrape):
            result = await scrape_assessment(page, assessment_id)
            all_results[assessment_id] = result
            
            status = result.get("status", "?")
            print(f"[{i+1}/{len(ids_to_scrape)}] ID {assessment_id}: {status}")
            
            # Save incrementally every 10 records
            if (i + 1) % 10 == 0 or i == len(ids_to_scrape) - 1:
                sorted_results = [all_results[k] for k in sorted(all_results)]
                with open(OUTPUT_FILE, "w") as f:
                    json.dump(sorted_results, f, indent=2, ensure_ascii=False)
                print(f"  -> Saved {len(all_results)} records to {OUTPUT_FILE}")
            
            await asyncio.sleep(REQUEST_DELAY)
        
        await browser.close()
    
    return [all_results[k] for k in sorted(all_results)]

In [11]:
# Run the scraper (set resume=True to continue from where you left off)
results = await run_scraper(start=START_ID, end=END_ID, resume=True)

Loaded 550 existing records, resuming...
IDs to scrape: 50
[1/50] ID 551: ok
[2/50] ID 552: ok
[3/50] ID 553: ok
[4/50] ID 554: ok
[5/50] ID 555: ok
[6/50] ID 556: ok
[7/50] ID 557: ok
[8/50] ID 558: ok
[9/50] ID 559: ok
[10/50] ID 560: ok
  -> Saved 560 records to data/iucn-rle-database/iucnrle_assessments.json
[11/50] ID 561: ok
[12/50] ID 562: ok
[13/50] ID 563: ok
[14/50] ID 564: ok
[15/50] ID 565: ok
[16/50] ID 566: ok
[17/50] ID 567: ok
[18/50] ID 568: ok
[19/50] ID 569: ok
[20/50] ID 570: ok
  -> Saved 570 records to data/iucn-rle-database/iucnrle_assessments.json
[21/50] ID 571: ok
[22/50] ID 572: ok
[23/50] ID 573: ok
[24/50] ID 574: ok
[25/50] ID 575: ok
[26/50] ID 576: ok
[27/50] ID 577: ok
[28/50] ID 578: ok
[29/50] ID 579: ok
[30/50] ID 580: ok
  -> Saved 580 records to data/iucn-rle-database/iucnrle_assessments.json
[31/50] ID 581: ok
[32/50] ID 582: ok
[33/50] ID 583: ok
[34/50] ID 584: ok
[35/50] ID 585: ok
[36/50] ID 586: ok
[37/50] ID 587: ok
[38/50] ID 588: ok
[39/50

In [ ]:
# ── Convert to DataFrame for easy inspection ──
df = pd.DataFrame(results)
print(f"Total records: {len(df)}")
print(f"Successful: {len(df[df['status'] == 'ok'])}")
print(f"Not found: {len(df[df['status'] == 'not_found'])}")
df[df['status'] == 'ok'].head(10)

In [12]:
import re
import ast
import json
import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path("data/iucn-rle-database")
LIST_COLS = ['iucn_habitats', 'iucn_typology', 'countries', 'taxa']

# ── Load data ──
json_path = OUTPUT_DIR / "iucnrle_assessments.json"
if json_path.exists():
    with open(json_path) as f:
        _data = json.load(f)
    df = pd.DataFrame(_data)
    print(f"Loaded {len(df)} records from {json_path}")
else:
    df = pd.DataFrame(results)
    print(f"Using in-memory results ({len(df)} records)")

print(f"  ok: {(df['status'] == 'ok').sum()}  |  not_found: {(df['status'] == 'not_found').sum()}")

# ── Clean habitat/typology item strings ──
# Removes leading "- ", strips spaces, then removes leading codes like:
# "5. ", "5.1. ", "F2. ", "F2.5 ", "TF1. ", "TF1.5 " etc.
_CODE_RE = re.compile(r'^[A-Z]{0,2}\d[\d.]*\.?\s+')

def clean_item(s):
    s = s.strip().lstrip('-').strip()
    s = _CODE_RE.sub('', s)
    return s

def clean_list_col(lst):
    if not isinstance(lst, list):
        return lst
    cleaned = [clean_item(x) for x in lst if x and x.strip().strip('-').strip()]
    # deduplicate preserving order
    seen = set()
    return [x for x in cleaned if x and not (x in seen or seen.add(x))]

def lists_to_str(df):
    out = df.copy()
    for col in LIST_COLS:
        if col in out.columns:
            out[col] = out[col].apply(
                lambda x: json.dumps(x, ensure_ascii=False) if isinstance(x, list) else x
            )
    return out

# ── Full CSV ──
df_ok = df[df['status'] == 'ok'].copy()
df_csv = lists_to_str(df_ok)
csv_path = OUTPUT_DIR / "iucnrle_assessments.csv"
df_csv.to_csv(csv_path, index=False)
print(f"Saved full CSV ({len(df_csv)} rows) → {csv_path}")

# ── Deduped CSV (by citation) ──
def merge_lists(series):
    combined = []
    for val in series:
        if isinstance(val, list):
            combined.extend(val)
        elif isinstance(val, str):
            stripped = val.strip()
            if stripped.startswith('['):
                try:
                    combined.extend(json.loads(stripped))
                    continue
                except Exception:
                    pass
            combined.append(val)
        elif val is not None:
            combined.append(str(val))
    seen = set()
    unique = [x for x in combined if not (x in seen or seen.add(x))]
    return json.dumps(unique, ensure_ascii=False)

def merge_scalar(series):
    vals = [str(v) for v in series if v is not None and str(v).strip()]
    seen = set()
    unique = [v for v in vals if not (v in seen or seen.add(v))]
    return json.dumps(unique, ensure_ascii=False) if len(unique) > 1 else (unique[0] if unique else None)

scalar_cols = [c for c in df_ok.columns if c not in LIST_COLS + ['citation']]
agg_rules = {col: merge_lists for col in LIST_COLS if col in df_ok.columns}
agg_rules.update({col: merge_scalar for col in scalar_cols if col in df_ok.columns})

df_deduped = df_ok.groupby('citation', as_index=False).agg(agg_rules)

# ── Clean iucn_habitats and iucn_typology in deduped df ──
for col in ['iucn_habitats', 'iucn_typology']:
    if col in df_deduped.columns:
        df_deduped[col] = df_deduped[col].apply(
            lambda x: json.dumps(
                clean_list_col(json.loads(x) if isinstance(x, str) and x.startswith('[') else x),
                ensure_ascii=False
            ) if x else x
        )

# ── id_len: number of original records merged into each deduped row ──
def parse_id_list(val):
    if val is None:
        return 1
    if isinstance(val, str) and val.startswith('['):
        try:
            return len(json.loads(val))
        except Exception:
            pass
    return 1

df_deduped['id_len'] = df_deduped['id'].apply(parse_id_list)

cols_order = ['citation', 'id_len'] + [c for c in df_deduped.columns if c not in ('citation', 'id_len')]
df_deduped = df_deduped[cols_order]

deduped_path = OUTPUT_DIR / "iucnrle_assessments_deduped.csv"
df_deduped.to_csv(deduped_path, index=False)
print(f"Saved deduped CSV ({len(df_deduped)} rows) → {deduped_path}")

Loaded 600 records from data/iucn-rle-database/iucnrle_assessments.json
  ok: 593  |  not_found: 0
Saved full CSV (593 rows) → data/iucn-rle-database/iucnrle_assessments.csv
Saved deduped CSV (33 rows) → data/iucn-rle-database/iucnrle_assessments_deduped.csv
